# src\api\axios.js:

In [ ]:
import axios from "axios"

const api = axios.create({

    baseURL: "http://127.0.0.1:8000/api/"

})

export default api  


# src\components\ProductCard.jsx:

In [ ]:
const ProductCard = ({ product, addToCart }) => {

    return (

        <div className="border rounded p-4 shadow">

            <img
                src={product.image}
                alt={product.title}
                style={{
                    width: "100%",
                    height: "200px",
                    objectFit: "cover"
                }}
            /> 

            <h2>{product.title}</h2>

            <p>{product.description}</p>

            <h3>${product.price}</h3>

            <button onClick={() => addToCart(product)}>
                Add To Cart
            </button>

        </div>
    );
};

export default ProductCard;


# src\hooks\useProducts.js:

In [ ]:
import { useEffect, useState } from "react"

import api from "../api/axios"

const useProducts = () => {

    const [products, setProducts] = useState([])
    const [loading, setLoading] = useState(true)

    useEffect(() => {

        api.get("products/")

            .then((response) => {

                console.log("DATOS BACKEND:")
                console.log(response.data)

                setProducts(response.data)

                setLoading(false)

            })

            .catch((error) => {

                console.log("ERROR AXIOS:")
                console.log(error)

                setLoading(false)

            })

    }, [])

    return {
        products,
        loading
    }
}

export default useProducts 

# src\pages\CartPages.jsx:

In [ ]:
import { useSelector, useDispatch } from "react-redux"

import { Link } from "react-router-dom"

import { removeFromCart } from "../redux/slices/cartSlice"

function CartPage() {

  const dispatch = useDispatch()

  const cart = useSelector(
    (state) => state.cart.items || []
  )

  return (

    <div style={{ padding: "40px" }}>

      <Link to="/">
        ← Volver
      </Link>

      <h1>Mi Carrito</h1>

      {

        cart.length === 0 ? (

          <h2>El carrito está vacío</h2>

        ) : (

          cart.map((item, index) => (

            <div
              key={index}
              style={{
                border: "1px solid #ccc",
                borderRadius: "10px",
                padding: "20px",
                marginBottom: "20px"
              }}
            >

              <h2>{item.title}</h2>

              <p>${item.price}</p>

              <button
                onClick={() => dispatch(removeFromCart(index))}
              >
                Eliminar Productos EBAC
              </button>

            </div>

          ))

        )

      }

    </div>

  )
}

export default CartPage


# src\pages\LoginPage.jsx:

In [ ]:
import { useState } from "react"

import { useDispatch } from "react-redux"

import { login } from "../redux/slices/authSlice"

import { useNavigate } from "react-router-dom"

function LoginPage() {

  const dispatch = useDispatch()

  const navigate = useNavigate()

  const [email, setEmail] = useState("")
  const [password, setPassword] = useState("")  

  const handleLogin = (e) => {

    e.preventDefault()

    const fakeUser = {

      email,

    }

    dispatch(login(fakeUser))

    navigate("/")

  }

  return (

    <div style={{ padding: "40px" }}>

      <h1>Login A EBAC</h1>

      <form onSubmit={handleLogin}>

        <input
          type="email"
          placeholder="Email"
          value={email}
          onChange={(e) => setEmail(e.target.value)}
        />

        <br />
        <br />

        <input
          type="password"
          placeholder="Password"
          value={password}
          onChange={(e) => setPassword(e.target.value)}
        />

        <br />
        <br />

        <button type="submit">
          Entrar
        </button>

      </form>

    </div>

  )
}

export default LoginPage


# src\redux\slices\authSlice.js:

In [ ]:
import { createSlice } from "@reduxjs/toolkit"

const loadUser = () => {

  try {

    const user = localStorage.getItem("user")

    return user ? JSON.parse(user) : null

  } catch (error) {

    return null

  }

}

const authSlice = createSlice({

  name: "auth",

  initialState: {

    user: loadUser(),

  },

  reducers: {

    login: (state, action) => {

      state.user = action.payload

      localStorage.setItem(
        "user",
        JSON.stringify(action.payload)
      )

    },

    logout: (state) => {

      state.user = null

      localStorage.removeItem("user")

    }

  }

})
 
export const {
  login,
  logout
} = authSlice.actions

export default authSlice.reducer


# src\redux\slices\cartSlice.js:

In [ ]:
import { createSlice } from "@reduxjs/toolkit";

const loadCartFromLocalStorage = () => {
  try {
    const serializedCart = localStorage.getItem("cart");
    return serializedCart ? JSON.parse(serializedCart) : [];
  } catch (error) {
    console.error("could not load cart from localStorage", error);
    return [];
  }
};

const saveCartToLocalStorage = (cart) => {
  try {
    const serializedCart = JSON.stringify(cart);
    localStorage.setItem("cart", serializedCart);
  } catch (error) {
    console.error("could not save cart to localStorage", error);
  }
};

const cartSlice = createSlice({
  name: "cart",

  initialState: {
    items: loadCartFromLocalStorage(),
  },

  reducers: {
    addToCart: (state, action) => {
      state.items.push(action.payload);
      saveCartToLocalStorage(state.items);
    },

    removeFromCart: (state, action) => {
      state.items.splice(action.payload, 1);
      saveCartToLocalStorage(state.items);
    },  

    clearCart: (state) => {
      state.items = [];
      saveCartToLocalStorage(state.items);
    },
  },
});
 
export const {
  addToCart,
  removeFromCart,
  clearCart,
} = cartSlice.actions;

export default cartSlice.reducer;


# src\redux\slices\orderSlice.js:

In [ ]:
import { createAsyncThunk, createSlice } from "@reduxjs/toolkit";
import api from "../../api/axios";
import { buildCreateApi } from "@reduxjs/toolkit/query";

const createOrder = createAsyncThunk("order/createOrder", async (order)=> {
    const response = await api.post("/orders", order);
    return response.data;
});

const orderSlice = createSlice({
    name: "order",
    initialState: {
        items: [],
        totalAmount: 0,
        status: "idle",
        error: null

    },

    reducers: {
        clearOrder: (state) => {
            state.items = [];
            state.totalAmount = 0;
        },
    },
    extraReducers: (builder) => {
        builder
            .addCase(createOrder.pending, (state) => {
                state.status = "loading";    
            })
            .addCase(createOrder.fulfilled, (state, action) => {
                state.status = "succeeded";
                state.items = action.payload.items;
                state.totalAmount = action.payload.totalAmount;
            })
            .addCase(createOrder.rejected, (state, action) => {
                state.status = "failed";
                state.error = action.error.messege;
            })
 

    }
    
});

export const { setOrder, clearOrder } = orderSlice.actions;
export default orderSlice.reducer;


# src\redux\slices\productSlice.js:

In [ ]:
import { createAsyncThunk, createSlice } from "@reduxjs/toolkit";
import api from "../../api/axios";

const fetchProducts = createAsyncThunk("products/fetchProducts", async () =>{
    const response = await api.get("/products");
    
    return response.data;
    
})
// idle \ pending \ fulfilled \ rejected
const productSlice = createSlice({
    name: "product",
    initialState: {
        items: [],
        status: "idle",
        error: "null"
    },
    reducers: {},
    extraReducers: (builder) => {
        builder
            .addCase(fetchProducts.pending, (state, action) => {
                state.status = "loading";
            })
            .addCase(fetchProducts.fulfilled, (state, action) => {
                state.status = "succeeded";
                state.items = action.payload;
            })
            .addCase(fetchProducts.rejected, (state, action) => {
                state.status = "failed";
                state.error = action.error.message;
            })

    }

});

export const { setProducts } = productSlice.actions;
export default productSlice.reducer;  


# src\redux\slices\userSlice.js:

In [ ]:
import { createAsyncThunk, createSlice } from "@reduxjs/toolkit";
import api from "../../api/axios";

const fetchUser = createAsyncThunk("user/fecthUser", async (userId) => {
    const response = await api.get(`/users/${userId}`);
    return response.data;
})

const createUser = createAsyncThunk("user/createUser", async (user) => {
    const response = await api.post("/users", {
      email: user.email,
      name: user.name,
      password: user.password,  
    });

    return response.data
})

const userSlice = createSlice({  

    name: "user",
    initialState: {
        currentUser : {
            email: "",
            name: "",
            
        }
    },
    reducers: {
        
        clearteUser: (state) => {
            state.currentUser = { email: "", name: "" };
        },
    },
    estraReducers: (build) => {
        build
            .addCase(fetchUser.pending, (state) => {
                state.state = "loading";
            })
            .addCase(fetchUser.fulfilled, (state, action) => {
                state.status = "succeeded";
                state.currentUser = action.payload
            })
            .addCase(fetchUser.rejected, (state, action) => {
                state.status = "failed";
                state.error = action.error.message;   
            })
            .addCase(createUser.pending, (state) => {
                state.state = "loading";
            })
            .addCase(createUser.fulfilled, (state, action) => {
                state.status = "succeeded";
                state.currentUser = action.payload
            })
            .addCase(createUser.rejected, (state, action) => {
                state.status = "failed";
                state.error = action.error.message;   
            })


    }
});


export const { login, logout } = userSlice.actions;
export default userSlice.reducer;



# src\redux\store.js:

In [ ]:
import { configureStore } from "@reduxjs/toolkit";
import authReducer from "./slices/authSlice"

import cartReducer from "./slices/cartSlice";
import userReducer from "./slices/userSlice";
import productReducer from "./slices/productSlice";
import orderReducer from "./slices/orderSlice";

const store = configureStore({
  reducer: {
    auth: authReducer,
    cart: cartReducer,
    user: userReducer,
    products: productReducer,
    order: orderReducer,
  },
});

export default store;   


# src\router\AppRouter.js:

In [ ]:
import React from "react";
import { BrowserRouter as Router, Route, Routes } from "react-router-dom";
import LogiPage from "../pages/LoginPage";
import LogiPage from "../pages/RegisterPage";
import HomePage from "../pages/HomePage";
import CartPage from "../pages/CartPage";
import CheckoutPage from "../pages/CheckoutPage";
import PostCheckout from "../pages/PostCheckoutPage";

const AppRouter = () => {
    return (
        <Router>
            <Routes>
                <Route path="/login" element={<LoginPage />} />
                <Route path="/register" element={<RegisterPage />} />
                <Route path="/" element={<HomePage />} />
                <Route path="/cart" element={<CartPage />} />
                <Route path="/checkout" element={<CheckoutPage />} />
                <Route path="/post-checkout" element={<PostCheckoutPage />} />
            </Routes>
        </Router>
    );
};

export default AppRouter  


# src\App.jsx:

In [ ]:
import { BrowserRouter, Routes, Route } from "react-router-dom"
import LoginPage from "./pages/LoginPage"
import ProductsPage from "./pages/ProductsPages"
import CartPage from "./pages/CartPages"

function App() {

  return (

    <BrowserRouter> 

      <Routes>

        <Route
          path="/"
          element={<ProductsPage />}
        />

        <Route
          path="/cart"
          element={<CartPage />}
        />

        <Route
          path="/login"
          element={<LoginPage />}
        />


      </Routes>

    </BrowserRouter>

  )
}

export default App


# src\component.jsx:

In [ ]:
import React from 'react'

export default function component() {
  return (
    <div>Mi Ecommerce</div>  
  )
}
